In [1]:
import sys
import json
from pathlib import Path

In [2]:
from transformers import AutoProcessor, AutoModelForVision2Seq

In [3]:
sys.path.append('..')

from agents import GenerateSpeaker
from interact_ui import run_sequential_games, run_reference_game

In [4]:
with open("coco_interact_tasks.json") as f:
    tasks = json.load(f)

In [50]:
model_type = "base_explicit"

In [6]:
processor = AutoProcessor.from_pretrained("saujasv/pixtral-12b")
model = AutoModelForVision2Seq.from_pretrained("saujasv/pixtral-12b", device_map="auto", torch_dtype="auto", attn_implementation={"text_config": "flash_attention_2"})
if model_type == "ft":
    model.load_adapter("saujasv/pixtral-vision_only-speaker_parts")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [51]:
agent = GenerateSpeaker(model, processor, '.', generation_config={"do_sample": True, "max_new_tokens": 64, "temperature": 0.8, "top_p": 0.95}, prompt_type="explicit" if "explicit" in model_type else "standard")

In [53]:
Path(f"interactions_coco/{model_type}/").mkdir(parents=True, exist_ok=True)
# _ = run_sequential_games(
#     agent,
#     [[f"../square-black-imgs/{img}.png" for img in t["image_set"]] for t in tasks],
#     [f"interactions/{model_type}/{t['task_id']}.json" for t in tasks],
#     [26 for t in tasks],
# )

In [13]:
import random

random.shuffle(tasks)

In [60]:
i = 3

In [61]:
print(tasks[i]['task_id'])

1006-8a964204-bce4-48de-9bb1-2933a4c017f7_2


In [62]:
_ = run_reference_game(agent, [f"/data/tir/projects/tir1/corpora/MSCOCO/images/{img}" for img in tasks[i]["image_set"]], f"interactions_coco/{model_type}/{tasks[i]['task_id']}.json", 16)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
_ = run_reference_game(agent, [f"../square-black-imgs/{img}.png" for img in tasks[i]["image_set"]], f"interactions/{model_type}/{tasks[i]['task_id']}.json", 25)

In [52]:
agent.prompt_type

'explicit'